In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from numpy.ma.extras import average

load_dotenv(override=True)
class agent:
    def __init__(self):
        self.model=init_chat_model(
            model="deepseek-v4-flash",
            model_provider="deepseek",
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            base_url=os.getenv("DEEPSEEK_BASE_URL"),
            temperature=0,
        )

llm=agent()

response=llm.model.invoke("你好")
print(response.content)


你好！很高兴见到你。有什么我可以帮你的吗？


tool

In [11]:
from langchain_core.tools import tool
@tool()
def get_menu(name:str)->str:
    """999"""
    menu={
        "拿铁":"28元",
        "美式":"21元",
        "蛋糕":"26元",
        "三明治":"12元"
    }
    return menu.get(name,"没有")
print(get_menu.invoke({
    "name":"三明治"
}))

12元


node

In [ ]:
from langgraph.graph import StateGraph,START,END
from state.coffee_state import CoffeeState
from tools.menu import get_menu
from tools.weather import get_weather

def classify_node(state: CoffeeState):
    query = state["query"]
    if "天气" in query:
        state["category"] = "weather"
    elif "多小钱" or "价格" in query:
        state["category"] = "menu"
    else:
        state["category"] = "normal"
    return state

def menu_node(state: CoffeeState):
    result=get_menu.invoke({
        "name":"拿铁",
    })
    state["response"]=result
    return state

def weather_node(state: CoffeeState):
    result=get_weather.invoke({
        "city":"北京",
    })
    state["response"]=result
    return state

def router(state: CoffeeState):
    if state["category"] == "menu":
        return "menu"
    if state["category"] == "weather":
        return "weather"
    return END

builder = StateGraph(CoffeeState)

builder.add_node(
    "classify",
    classify_node
)
builder.add_node(
    "menu",
    menu_node
)
builder.add_node(
    "weather",
    weather_node
)

builder.add_edge(
    START,
    "classify",
)
builder.add_conditional_edges(
    "classify",
    router,{
        "menu":"menu",
        "weather":"weather",
        END:END
    }
)
builder.add_edge(
    "menu",
    END
)
builder.add_edge(
    "weather",
    END
)
graph=builder.conpile()


In [ ]:
from langgraph.graph import StateGraph,START,END
from graph.state import TravelState

from agents.research_agent import create_research_agent
from agents.planner_agent import create_planner_agent
from agents.writer_agent import create_writer_agent
from agents.supervisor import create_supervisor_agent
from llm import LLM

model=LLM()
research_agent = create_research_agent(model)
planner_agent = create_planner_agent(model)
writer_agent = create_writer_agent(model)
supervisor_agent = create_supervisor_agent(model)

def supervisor_node(state: TravelState):
    state["step_count"]+=1
    prompt=f"""
用户需求：

{state["user_query"]}

Research Agent 的结果：

{state["research_result"]}

Planner Agent 的结果：

{state["plan_result"]}

Writer Agent 的结果：

{state["final_result"]}

请判断下一步应该执行哪个 Agent。

只能返回：

research
planner
writer
FINISH
"""
    result = supervisor_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    })
    decision = result["messages"][-1].content.strip()
    state["current_agent"] = decision
    print(f"[Supervisor] {decision}")
    return state

def research_node(state: TravelState):
    result=research_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":state["user_query"]
            }
        ]
    })
    state["research_result"] = (
        result["messages"][-1].content
    )
    return state

def planner_node(state: TravelState):
    prompt=f"""
    用户需求：

{state["user_query"]}

Research Agent 的研究结果：

{state["research_result"]}

请根据这些信息制定旅行计划。
"""
    result=planner_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":prompt
            }
        ]
    })
    state["plan_result"] = (
        result["messages"][-1].content
    )
    return state

def writer_node(state: TravelState):
    prompt=f"""
用户请求:

{state["user_query"]}

研究结果:

{state["research_result"]}

计划旅行:

{state["plan_result"]}

请整理成最终旅行方案
"""
    result=writer_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":prompt
            }
        ]
    })
    state["final_result"] = (
        result["messages"][-1].content
    )
    return state

def router_supervisor(state:TravelState):
    if state["step_count"]>=10:
        return "FINISH"
    decision = state["current_agent"]
    allowed = {
        "research",
        "planner",
        "writer",
        "FINISH"
    }
    if decision not in allowed:
        raise ValueError(
            f"非法 Supervisor decision: {decision}"
        )
    return decision


builder=StateGraph(TravelState)

builder.add_node(
    "supervisor",
    supervisor_node
)
builder.add_node(
    "research_agent",
    research_node
)
builder.add_node(
    "planner_agent",
    planner_node
)
builder.add_node(
    "writer_agent",
    writer_node
)

builder.add_edge(
    START,
    "supervisor",
)
builder.add_conditional_edges(
    "supervisor",
    router_supervisor,
    {
        "research":"research_agent",
        "planner":"planner_agent",
        "writer":"writer_agent",
        "supervisor":"supervisor",
        "FINISH":END
    }
)
builder.add_edge(
    "research_agent",
    "supervisor",
)
builder.add_edge(
    "planner_agent",
    "supervisor",
)
builder.add_edge(
    "writer_agent",
    "supervisor",
)


graph=builder.compile()

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """
    查询城市天气。
    """

    try:

        weather_data = {
            "东京": "晴天，25°C",
            "京都": "多云，23°C",
            "大阪": "小雨，21°C",
        }

        # 模拟天气服务故障
        if city == "故障测试":
            raise RuntimeError("天气服务暂时不可用")

        return weather_data.get(
            city,
            f"暂时没有{city}的天气数据"
        )

    except Exception as e:

        print(f"[Weather Error] {e}")

        return "天气查询失败，请稍后再试。"

In [ ]:
from langgraph.graph import StateGraph,START,END
from graph.state import TravelState

from agents.research_agent import create_research_agent
from agents.planner_agent import create_planner_agent
from agents.writer_agent import create_writer_agent
from agents.supervisor import create_supervisor_agent

from utils.context_builder import builder_supervisor_context,builder_research_context,builder_planner_context,builder_writer_context
from utils.supervisor_validator import (normalize_decision,validate_decision)
from llm import LLM

model=LLM()
research_agent = create_research_agent(model)
planner_agent = create_planner_agent(model)
writer_agent = create_writer_agent(model)
supervisor_agent = create_supervisor_agent(model)



def supervisor_node(state: TravelState):
    state["step_count"]=0
    state["step_count"]+=1
    prompt=builder_supervisor_context(state)
    result = supervisor_agent.invoke({
        "messages":[
        {
            "role":"user",
            "content":prompt
        }
    ]
    })
    raw_decision = result["messages"][-1].content.strip()
    decision = normalize_decision(raw_decision)
    validate_decision(state,decision)
    state["current_agent"]=decision
    print(f"[Supervisor] {result}")
    return state

def research_node(state: TravelState):
    result=research_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":state["user_query"]
            }
        ]
    })
    state["research_result"] = (
        result["messages"][-1].content
    )
    return state

def planner_node(state: TravelState):
    prompt=builder_planner_context(state)
    result=planner_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":prompt
            }
        ]
    })
    state["plan_result"] = (
        result["messages"][-1].content
    )
    return state

def writer_node(state: TravelState):
    prompt=builder_writer_context(state)
    result=writer_agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":prompt
            }
        ]
    })
    state["final_result"] = (
        result["messages"][-1].content
    )
    return state

def router_supervisor(state:TravelState):
    if state["step_count"]>=10:
        return "FINISH"
    return state["current_agent"]

builder=StateGraph(TravelState)

builder.add_node(
    "supervisor",
    supervisor_node
)
builder.add_node(
    "research_agent",
    research_node
)
builder.add_node(
    "planner_agent",
    planner_node
)
builder.add_node(
    "writer_agent",
    writer_node
)

builder.add_edge(
    START,
    "supervisor",
)
builder.add_conditional_edges(
    "supervisor",
    router_supervisor,
    {
        "research":"research_agent",
        "planner":"planner_agent",
        "writer":"writer_agent",
        "FINISH":END
    }
)
builder.add_edge(
    "research_agent",
    "supervisor",
)
builder.add_edge(
    "planner_agent",
    "supervisor",
)
builder.add_edge(
    "writer_agent",
    "supervisor",
)


graph=builder.compile()

In [ ]:
#002